# Late-Delivery Predictor for an E-Commerce Marketplace

**Dataset:** Olist Brazilian E-Commerce Public Dataset (Kaggle) · **Goal:** predict, at purchase time, whether an order will arrive after its estimated delivery date.

**How to run:** select the project's `.venv` kernel (top right), then **Run → Restart & Run All**. The notebook runs top to bottom (roughly 10 to 20 minutes, mostly the hyperparameter search; not timed on your machine) and writes the model files and the data files used by the Streamlit app.

| Section | Requirement |
|---|---|
| 1 | Problem framing |
| 2 | Data understanding |
| 3 | Exploratory data analysis |
| 4 | Data preprocessing |
| 5 | Machine learning modeling |
| 6 | Tuning and validation |
| 7 | Model evaluation on the untouched test set |
| 8 | Business insights |
| 9 | Files exported for the Streamlit application |

## 1. Problem framing

**Business problem.** A marketplace promises every customer an estimated delivery date. When an order arrives after that date, the customer is disappointed, support tickets rise and repeat purchases suffer. Today the marketplace learns about a late delivery only after it has happened.

**Who faces it.** Operations and customer-service teams (who must react), sellers (whose handling time drives delays) and customers (who receive the order late).

**Decision the model supports.** Which orders deserve proactive attention *right after purchase*, for example contacting the customer early, pushing the seller to ship faster, choosing a faster carrier or resetting the promised date. The model ranks orders by risk so the limited attention of the operations team goes to the orders most likely to be late.

**Target variable.**
- `is_late = 1` if the customer received the order on a calendar day **after** the estimated delivery date.
- `is_late = 0` if the order arrived on or before it.
- Only orders with status *delivered* and both dates present can be labelled. Canceled, unavailable and in-transit orders have no outcome yet and are excluded.

**Evaluation metrics and why they matter.**

| Metric | Why it matters here |
|---|---|
| PR-AUC (average precision) | Main metric. Late orders are rare, and PR-AUC measures how well the model ranks them above on-time orders. |
| Recall in the riskiest 10% of orders | Business reading: "if we act on 10% of orders, what share of all late deliveries do we catch?" |
| Precision / false alarms per catch | Every false alarm costs operations time. |
| F1-score | Balances precision and recall at a chosen cut-off. |
| ROC-AUC | Overall ranking quality, less informative with rare positives. |
| Accuracy | **Not used as a decision metric.** A model that always says "on time" would already score about 93%. |

**Leakage rule.** Only information available **at purchase time** may be a model input. Anything that happens after purchase (approval time, carrier hand-over, delivery date, order status, reviews) is used only to build the target and for analysis.

In [ ]:
import sys
import json
import time
import platform
import warnings
from importlib.metadata import version
from pathlib import Path

# Find the project root (the folder that contains `src`), wherever the notebook is opened from
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, brier_score_loss,
    classification_report, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)

from src.config import DATA_PROCESSED, FIGURES_DIR, MODEL_PATH, MODELS_DIR, RANDOM_STATE
from src.data_loader import load_all
from src.build_dataset import (
    ANALYSIS_ONLY, BASE_PATH, FEATURE_COLUMNS, ID_COLUMNS, NUMERIC_FEATURES, TARGET,
    build_order_table, load_base_table,
)
from src.preprocessing import (
    MODEL_FEATURES, NUMERIC_MODEL, TEST_START, build_preprocessor, split_xy,
    time_based_split,
)
from src.models import MODEL_NAMES, build_pipeline
from src.validation import FOLD_CUTOFFS, evaluate_scores, make_time_folds, top_k_metrics
from src.tuning import SCORING, SEARCH_SPACES, TUNED_MODELS, base_pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
BLUE = "#4C72B0"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## 2. Data understanding

The Olist dataset has nine CSV files. Each table is loaded with its date columns parsed as datetimes. The tables are linked like this:

```
orders ──customer_id──> customers ──zip prefix──> geolocation
  │ order_id
  ├── items ──product_id──> products ──category──> category_translation
  │          └─ seller_id──> sellers ──zip prefix──> geolocation
  ├── payments   (one or more rows per order)
  └── reviews    (written after delivery)
```

In [ ]:
tables = load_all()

overview = pd.DataFrame([{
    "table": name,
    "rows": len(d),
    "columns": d.shape[1],
    "memory_MB": round(d.memory_usage(deep=True).sum() / 1_048_576, 1),
    "duplicate_rows": int(d.duplicated().sum()),
} for name, d in tables.items()])
display(overview)

**Columns, data types and missing values.** Only columns that actually contain missing values are listed.

In [ ]:
for name in ["orders", "items", "customers", "sellers", "products"]:
    print(f"--- {name}: {len(tables[name]):,} rows ---")
    display(pd.DataFrame({"dtype": tables[name].dtypes.astype(str),
                          "missing": tables[name].isna().sum(),
                          "unique": tables[name].nunique()}))

miss = []
for name, d in tables.items():
    for col in d.columns:
        k = int(d[col].isna().sum())
        if k:
            miss.append({"table": name, "column": col, "dtype": str(d[col].dtype),
                         "missing": k, "missing_pct": round(k / len(d) * 100, 2)})
print("Columns with missing values (all tables):")
display(pd.DataFrame(miss))

**Do the join keys really match?** Every relationship in the diagram is measured instead of assumed.

In [ ]:
links = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("products", "product_category_name", "category_translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]
rel = pd.DataFrame([{
    "child key": f"{c}.{cc}",
    "parent key": f"{p}.{pc}",
    "matched_%": round(tables[c][cc].dropna().isin(tables[p][pc]).mean() * 100, 2),
    "parent key unique": tables[p][pc].is_unique,
} for c, cc, p, pc in links])
display(rel)

orders, items = tables["orders"], tables["items"]
per_order = items.groupby("order_id").agg(n_items=("order_item_id", "size"),
                                          n_sellers=("seller_id", "nunique"))
print(f"Orders without item rows: {int((~orders['order_id'].isin(items['order_id'])).sum()):,}")
print(f"Orders with more than one seller: {int((per_order['n_sellers'] > 1).sum()):,}")
print(f"Max items in one order: {per_order['n_items'].max()}")
print(f"Geolocation duplicate rows: {int(tables['geolocation'].duplicated().sum()):,} "
      "(the zip code key is not unique, so it must be collapsed before joining)")

**Order status, date range and class balance.** The target needs a delivered order with both a delivery date and an estimated date. The estimated date has no time of day, so "late" is judged on calendar days.

In [ ]:
status = orders["order_status"].value_counts().to_frame("orders")
status["pct"] = (status["orders"] / len(orders) * 100).round(2)
display(status)
print("Purchase dates:", orders["order_purchase_timestamp"].min(), "->",
      orders["order_purchase_timestamp"].max())

delivered = orders[orders["order_status"] == "delivered"].dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
d_day = delivered["order_delivered_customer_date"].dt.normalize()
e_day = delivered["order_estimated_delivery_date"].dt.normalize()
late_day = d_day > e_day
late_exact = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
print(f"\nLabelled orders: {len(delivered):,}")
print(f"Late, calendar-day comparison (chosen): {late_day.sum():,} ({late_day.mean() * 100:.2f}%)")
print(f"Late, exact timestamp comparison:       {late_exact.sum():,} ({late_exact.mean() * 100:.2f}%)")

**Build one row per order.** Items, products, sellers, payments and the collapsed geolocation table are aggregated into a single order-level table (`src/build_dataset.py`). Every column is placed in one of three groups:

- `TARGET`: what we predict.
- `FEATURE_COLUMNS`: known at purchase time, allowed as model inputs.
- `ANALYSIS_ONLY`: known only after purchase, used for analysis and error analysis, **never as a feature**.

In [ ]:
df = build_order_table()
df.to_csv(BASE_PATH, index=False)
df = load_base_table()          # reload so dates are parsed exactly as the other modules expect

counts = df[TARGET].value_counts().sort_index()
print(f"\nRows: {len(df):,}   Columns: {df.shape[1]}   order_id unique: {df['order_id'].is_unique}")
print(f"On-time (0): {counts[0]:,}   Late (1): {counts[1]:,}   Late rate: {df[TARGET].mean() * 100:.2f}%")

m = df[FEATURE_COLUMNS].isna().sum()
print("\nMissing values in purchase-time features:")
display(m[m > 0].to_frame("missing"))

print("Allowed model features:", FEATURE_COLUMNS)
print("Analysis-only (never features):", ANALYSIS_ONLY)
print("Overlap:", set(FEATURE_COLUMNS) & set(ANALYSIS_ONLY) or "none")

## 3. Exploratory data analysis

The whole labelled table is used here **only for reading** patterns. Nothing is fitted on it. Every statistic that drives a modelling decision is recomputed on the training split in section 4.

In [ ]:
OVERALL = df[TARGET].mean() * 100


def late_rate(data, col, min_orders=1):
    g = data.groupby(col, observed=True)[TARGET].agg(orders="size", late_rate_pct="mean")
    g["late_rate_pct"] = (g["late_rate_pct"] * 100).round(2)
    return g[g["orders"] >= min_orders]


def finish(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=130)
    plt.show()


def barh_rate(tab, title, name, top=None):
    t = tab.sort_values("late_rate_pct", ascending=False)
    if top:
        t = t.head(top)
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.32 * len(t) + 1.5)))
    sns.barplot(x=t["late_rate_pct"], y=t.index.astype(str), color=BLUE, ax=ax, orient="h")
    ax.axvline(OVERALL, color="red", ls="--", label=f"overall {OVERALL:.1f}%")
    ax.set_xlabel("Late rate (%)")
    ax.set_title(title)
    ax.legend()
    finish(fig, name)


def bar_rate(tab, title, name):
    t = tab.sort_index()
    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.barplot(x=t.index.astype(str), y=t["late_rate_pct"], color=BLUE, ax=ax)
    ax.axhline(OVERALL, color="red", ls="--", label=f"overall {OVERALL:.1f}%")
    ax.set_ylabel("Late rate (%)")
    ax.set_title(title)
    ax.legend()
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
    finish(fig, name)

### 3.1 Univariate analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=["On-time (0)", "Late (1)"], y=counts.values, color=BLUE, ax=ax)
ax.set_title("Target distribution")
ax.set_ylabel("Orders")
finish(fig, "01_target_distribution.png")

numeric = ["estimated_delivery_days", "distance_km", "total_price", "total_freight",
           "freight_ratio", "total_weight_g", "total_volume_cm3", "shipping_limit_days"]
display(df[numeric].describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(2).T)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, col in zip(axes.ravel(), numeric):
    s = df[col].dropna()
    sns.histplot(s[s <= s.quantile(0.99)], bins=40, ax=ax, color=BLUE)
    ax.set_title(col)
fig.suptitle("Numeric features (values above the 99th percentile hidden)")
finish(fig, "02_numeric_distributions.png")

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(df["days_vs_estimate"].clip(-40, 40), bins=80, ax=ax, color=BLUE)
ax.axvline(0, color="red", ls="--", label="estimated date (right of line = late)")
ax.set_title("Actual minus estimated delivery date, in days (clipped at +/-40)")
ax.legend()
finish(fig, "03_days_vs_estimate.png")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col, n) in zip(axes.ravel(), [("customer_state", 10), ("seller_state", 10),
                                       ("product_category", 12), ("payment_type", 6)]):
    vc = df[col].value_counts().head(n)
    sns.barplot(x=vc.values, y=vc.index.astype(str), color=BLUE, ax=ax, orient="h")
    ax.set_title(f"Top {len(vc)}: {col}")
finish(fig, "04_categorical_counts.png")

### 3.2 Bivariate analysis: late rate by factor

In [ ]:
tab = late_rate(df, "customer_state", 200)
display(tab.sort_values("late_rate_pct", ascending=False).head(10))
barh_rate(tab, "Late rate by customer state (>= 200 orders)", "05_late_by_customer_state.png")

tab = late_rate(df, "seller_state", 100)
display(tab.sort_values("late_rate_pct", ascending=False).head(10))
barh_rate(tab, "Late rate by seller state (>= 100 orders)", "06_late_by_seller_state.png")

tab = late_rate(df, "product_category", 300)
display(tab.sort_values("late_rate_pct", ascending=False).head(10))
barh_rate(tab, "Late rate by product category (>= 300 orders, top 15)",
          "07_late_by_category.png", top=15)

tab = late_rate(df, "purchase_year_month", 200)
display(tab)
bar_rate(tab, "Late rate by purchase month", "08_late_by_year_month.png")

In [ ]:
for col, fname in [("distance_km", "12_late_by_distance.png"),
                   ("estimated_delivery_days", "13_late_by_estimated_days.png"),
                   ("total_freight", "14_late_by_freight.png")]:
    bins, edges = pd.qcut(df[col], 10, labels=False, retbins=True, duplicates="drop")
    tab = late_rate(df.assign(_decile=bins + 1), "_decile")
    print(f"{col}: decile edges {np.round(edges, 1).tolist()}")
    display(tab.T)
    bar_rate(tab, f"Late rate by {col} decile (1 = lowest)", fname)

tmp = df.assign(n_sellers_capped=df["n_sellers"].clip(upper=3))
for col, title, fname in [
    ("same_state", "Late rate: seller and customer in the same state (1 = yes)", "18_late_by_same_state.png"),
    ("payment_type", "Late rate by payment type", "17_late_by_payment.png"),
    ("n_sellers_capped", "Late rate by number of sellers (3 = 3 or more)", "19_late_by_n_sellers.png"),
]:
    tab = late_rate(tmp, col, 30)
    display(tab.T)
    bar_rate(tab, title, fname)

r = df.dropna(subset=["review_score"])
print("Customer impact (review scores exist only after delivery, so this is analysis only):")
display(r.groupby(TARGET)["review_score"].agg(orders="size", mean_score="mean").round(2))
print("Share of 1-2 star reviews (%):")
display((r.assign(low=r["review_score"] <= 2).groupby(TARGET)["low"].mean() * 100).round(2).to_frame("low_review_pct"))

### 3.3 Multivariate analysis

In [ ]:
corr = df[NUMERIC_FEATURES + [TARGET]].corr(numeric_only=True)
display(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3).to_frame("corr_with_is_late"))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", annot_kws={"size": 7}, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix (purchase-time numeric features + target)")
finish(fig, "22_correlation_heatmap.png")

d_bin, d_edges = pd.qcut(df["distance_km"], 5, labels=False, retbins=True, duplicates="drop")
e_bin, e_edges = pd.qcut(df["estimated_delivery_days"], 5, labels=False, retbins=True, duplicates="drop")
print("Distance quintile edges (km):", np.round(d_edges, 0).tolist())
print("Promised-days quintile edges:", np.round(e_edges, 0).tolist())
pivot = df.assign(dist_q=d_bin + 1, est_q=e_bin + 1).pivot_table(
    index="dist_q", columns="est_q", values=TARGET, aggfunc="mean") * 100
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
ax.set_xlabel("Promised delivery days (quintile, 1 = shortest)")
ax.set_ylabel("Distance (quintile, 1 = closest)")
ax.set_title("Late rate (%)")
finish(fig, "23_distance_x_promised_days.png")

top_c = df["customer_state"].value_counts().head(8).index
top_s = df["seller_state"].value_counts().head(8).index
sub = df[df["customer_state"].isin(top_c) & df["seller_state"].isin(top_s)]
cnt = pd.crosstab(sub["seller_state"], sub["customer_state"])
rate = pd.crosstab(sub["seller_state"], sub["customer_state"], values=sub[TARGET], aggfunc="mean") * 100
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(rate.where(cnt >= 30), annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
ax.set_title("Late rate (%) by route (cells with < 30 orders hidden)")
finish(fig, "24_route_heatmap.png")

seller = df.groupby("seller_id")[TARGET].agg(orders="size", late_orders="sum")
seller["late_rate_pct"] = (seller["late_orders"] / seller["orders"] * 100).round(2)
big = seller[seller["orders"] >= 30]
print(f"Sellers: {len(seller):,}; with >= 30 orders: {len(big):,}")
display(big["late_rate_pct"].describe().round(2).to_frame("late_rate_pct"))
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(big["late_rate_pct"], bins=30, color=BLUE, ax=ax)
ax.axvline(OVERALL, color="red", ls="--", label=f"overall {OVERALL:.1f}%")
ax.set_title("Seller late rate (sellers with >= 30 orders)")
ax.set_xlabel("Late rate (%)")
ax.legend()
finish(fig, "25_seller_late_rate_distribution.png")

### 3.4 Insights and hypotheses from the findings

1. **Lateness is mostly a distance and route problem.** The farthest distance decile is late 11.7% of the time against 4.2% for the closest. Cross-state orders are late 8.1% of the time against 4.5% for same-state orders. Northern and north-eastern customer states are worst (AL 21.4%, MA 17.4%, SE 15.2%, CE 13.8%), and RJ combines a high rate (12.1%) with a large volume.
2. **Delays come in waves, not in a stable seasonal pattern.** November 2017 was 12.4% late, February 2018 14.1% and March 2018 19.0%, while June 2018 was only 1.2%. The same calendar month behaves very differently in different years (March: 4.6% in 2017, 19.0% in 2018). Hypothesis: the November peak is linked to Black Friday volume; the Feb–Mar 2018 spike looks like a one-off operational disruption, because months with similar volume were not late.
3. **The promised delivery window is not scaled to distance.** For the farthest orders the late rate is about 14.9% when 17–22 days are promised and 6.1% when 31 days or more are promised. Hypothesis: promises are too tight for far deliveries, an interaction a linear model cannot capture without help.
4. **Seller reliability varies widely.** Among sellers with at least 30 orders the median late rate is 5.6%, the 75th percentile 9.1% and the worst ten sellers reach 20.6–34.3%. These rates rest on 35–94 orders each, so part of the spread is noise; we check this on held-out data in section 8.
5. **Late delivery hurts satisfaction directly.** Late orders average a review score of 2.27 against 4.29, and 62.4% of them get 1–2 stars against 9.2% for on-time orders.

Weak factors: weekday, hour of day and payment type barely move the late rate, and no numeric feature has a correlation with lateness above 0.08. We should therefore expect a useful but modest model.

## 4. Data preprocessing

**Time-based split instead of a random split.** Late orders cluster in specific weeks, and the deployed model must predict *future* orders. So orders purchased before **2018-06-01** are used for training and orders from that date on form the untouched test set. Training orders that were bought before the cut-off but delivered after it are **purged**, because their label would not be known yet on the cut-off date.

**Pipeline decisions** (all fitted on training data only, inside a scikit-learn `Pipeline`):
- `purchase_month` is dropped: each calendar month is seen only once or twice and behaves differently each year, so the model would memorise one-off events.
- Missing numeric values: median imputation. Missing categories: a `"missing"` label.
- Outliers: each numeric feature is clipped at the 1st and 99th percentile learned from the training data (custom `QuantileClipper`).
- Skewed money, weight and volume features get `log1p`; all numeric features are standardised.
- Categorical features are one-hot encoded; categories with fewer than 100 training orders share one "infrequent" bucket, and unseen categories do not crash prediction.
- Seller identity is **not** a feature (a seller's late rate would have to be computed from training labels only). Duplicates: none exist at order level.

In [ ]:
train, test, purged = time_based_split(df)
X_train, y_train = split_xy(train)
X_test, y_test = split_xy(test)

print(f"Duplicated order_id: {int(df['order_id'].duplicated().sum())}   "
      f"Fully duplicated rows: {int(df.duplicated().sum())}")
print(f"Purged from training (bought before the cut-off, delivered after it): {purged:,}")
display(pd.DataFrame([{
    "set": name, "orders": len(p), "late": int(p[TARGET].sum()),
    "late_rate_pct": round(p[TARGET].mean() * 100, 2),
    "first_purchase": p["order_purchase_timestamp"].min().date(),
    "last_purchase": p["order_purchase_timestamp"].max().date(),
} for name, p in (("train", train), ("test", test))]))

checks = {
    "no analysis-only column is a model feature": not set(MODEL_FEATURES) & set(ANALYSIS_ONLY),
    "no id column is a model feature": not set(MODEL_FEATURES) & set(ID_COLUMNS),
    "target is not a model feature": TARGET not in MODEL_FEATURES,
    "every training purchase is before the test period":
        train["order_purchase_timestamp"].max() < test["order_purchase_timestamp"].min(),
    "every training delivery happened before the test period":
        train["order_delivered_customer_date"].max() < TEST_START,
    "no order appears in both sets": not set(train["order_id"]) & set(test["order_id"]),
}
for name, ok in checks.items():
    print(f"[{'OK' if ok else 'FAIL'}] {name}")
assert all(checks.values()), "A leakage check failed"
print(f"\nModel features ({len(MODEL_FEATURES)}): {MODEL_FEATURES}")

**Why is the late rate in the training data lower than in validation windows?** Late orders take longer to arrive, so on a given cut-off date more of them are still in transit and have no label yet. The table below checks this. The "in transit" labels are shown for this diagnostic only and are never used for training.

In [ ]:
bought = df["order_purchase_timestamp"]
rows = []
for c in FOLD_CUTOFFS + [TEST_START]:
    before = df[bought < c]
    known = before[before["order_delivered_customer_date"] < c]
    transit = before[before["order_delivered_customer_date"] >= c]
    rows.append({
        "cutoff": c.date(), "bought_before": len(before),
        "late_pct_label_known": round(known[TARGET].mean() * 100, 2),
        "in_transit": len(transit),
        "late_pct_in_transit": round(transit[TARGET].mean() * 100, 2),
    })
display(pd.DataFrame(rows))

**The preprocessor, fitted on the training set only.** The test set is only transformed, using the training statistics.

In [ ]:
pre = build_preprocessor()
display(pre)
Xt_train = pre.fit_transform(X_train)     # fit on TRAIN only
Xt_test = pre.transform(X_test)           # TEST is only transformed
print(f"Input features: {X_train.shape[1]}  ->  transformed columns: {Xt_train.shape[1]}")
print(f"Train: {Xt_train.shape}   Test: {Xt_test.shape}")
print(f"NaN left in train: {int(np.isnan(Xt_train).sum())}   in test: {int(np.isnan(Xt_test).sum())}")

## 5. Machine learning modeling

Four models are compared, each wrapped in a `Pipeline` (preprocessor + model) so that preprocessing is refitted inside every validation fold:

| Model | What it is | Strength | Weakness |
|---|---|---|---|
| **Dummy (prior)** | Always outputs the training late rate | The floor every model must beat | Learns nothing |
| **Logistic Regression** | Linear model of the log-odds of being late | Simple, fast, explainable | Cannot capture interactions by itself |
| **Random Forest** | 300 averaged decision trees | Finds non-linear effects and interactions | Slower, larger saved file |
| **XGBoost** | Small trees built one after another, each correcting the previous | Often the strongest on tabular data | Needs tuning |

**Imbalance** (about 7% late) is handled with class weights (`class_weight`, `scale_pos_weight`), not with synthetic samples.

**Validation.** Three expanding time windows *inside the training set* (validation starts 2017-09-01, 2017-12-01 and 2018-03-01). Each fold is fitted only on orders bought **and delivered** before its window. **The test set is not used in sections 5 and 6.**

In [ ]:
folds = make_time_folds(train)
fold_bought = train["order_purchase_timestamp"]
display(pd.DataFrame([{
    "fold": i, "fit_orders": len(f), "fit_late_pct": round(y_train.iloc[f].mean() * 100, 2),
    "val_orders": len(v), "val_late_pct": round(y_train.iloc[v].mean() * 100, 2),
    "val_from": fold_bought.iloc[v].min().date(), "val_to": fold_bought.iloc[v].max().date(),
} for i, (f, v) in enumerate(folds, 1)]))

records = []
for name in MODEL_NAMES:
    for i, (fit_idx, val_idx) in enumerate(folds, 1):
        pipe = build_pipeline(name, y_train.iloc[fit_idx])
        pipe.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
        proba = pipe.predict_proba(X_train.iloc[val_idx])[:, 1]
        rec = {"model": name, "fold": i}
        rec.update(evaluate_scores(y_train.iloc[val_idx], proba))
        records.append(rec)
    print(f"done: {name}")
cv5 = pd.DataFrame(records)
cv5["pr_auc_x_baseline"] = cv5["pr_auc"] / cv5["prevalence"]
display(cv5[["model", "fold", "prevalence", "pr_auc", "pr_auc_x_baseline", "roc_auc",
             "recall_top10", "lift_top10", "f1_05"]].round(3))

In [ ]:
cols = ["pr_auc", "pr_auc_x_baseline", "roc_auc", "recall_top10", "precision_top10", "lift_top10", "f1_05"]
order = cv5.groupby("model")["pr_auc"].mean().sort_values(ascending=False).index
summary = pd.DataFrame({
    m: {c: f"{cv5[cv5.model == m][c].mean():.3f} +/- {cv5[cv5.model == m][c].std():.3f}" for c in cols}
    for m in order}).T
display(summary)

means = cv5.groupby("model")[["pr_auc", "recall_top10"]].agg(["mean", "std"]).loc[MODEL_NAMES]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (c, title) in zip(axes, [("pr_auc", "PR-AUC (mean across folds)"),
                                 ("recall_top10", "Recall within the riskiest 10% of orders")]):
    ax.bar(MODEL_NAMES, means[(c, "mean")], yerr=means[(c, "std")], capsize=5, color=BLUE)
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
finish(fig, "26_model_comparison.png")

**How to read this table.** `pr_auc_x_baseline` divides PR-AUC by the late rate of the validation fold: 1.0 means no better than guessing, which is exactly where the Dummy sits. A real model must clearly beat the Dummy *and* the fold-to-fold spread (the "+/-"), because the validation windows contain very different periods (for example the November 2017 and March 2018 delay waves).

## 6. Tuning and validation

- `GridSearchCV` for Logistic Regression (12 combinations) and `RandomizedSearchCV` for Random Forest (12 candidates) and XGBoost (30 candidates).
- The **same time-ordered folds** are used and the preprocessor is refitted inside each fold. **Only training data is used.**
- The imbalance weight `scale_pos_weight` is a tuned hyperparameter, so no labels outside the fold are used to set it.

**Selection rule (declared before seeing tuned results):** keep every model whose mean CV PR-AUC is within 0.01 of the best, then choose the one with the highest recall in the riskiest 10% of orders. I chose this because default-model gaps were smaller than the fold-to-fold noise, so a rule based on PR-AUC alone would choose on noise.

In [ ]:
TIE_MARGIN = 0.01
searches, rows, all_cv = {}, [], []


def py(v):
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return round(float(v), 4)
    return v


for name in TUNED_MODELS:
    kind, space, n_iter = SEARCH_SPACES[name]
    common = dict(scoring=SCORING, refit="pr_auc", cv=folds, n_jobs=1, error_score="raise")
    if kind == "grid":
        n_cand = int(np.prod([len(v) for v in space.values()]))
        search = GridSearchCV(base_pipeline(name), space, **common)
    else:
        n_cand = n_iter
        search = RandomizedSearchCV(base_pipeline(name), space, n_iter=n_iter,
                                    random_state=RANDOM_STATE, **common)
    print(f"Tuning {name}: {n_cand} candidates x {len(folds)} folds ...")
    t0 = time.time()
    search.fit(X_train, y_train)
    print(f"  done in {(time.time() - t0) / 60:.1f} min. Best CV PR-AUC: {search.best_score_:.3f}")
    cv = search.cv_results_
    i = search.best_index_
    rows.append({
        "model": name,
        "pr_auc": cv["mean_test_pr_auc"][i], "pr_auc_std": cv["std_test_pr_auc"][i],
        "roc_auc": cv["mean_test_roc_auc"][i],
        "recall_top10": cv["mean_test_recall_top10"][i],
        "fold_pr_auc": [round(cv[f"split{k}_test_pr_auc"][i], 3) for k in range(len(folds))],
        "best_params": {k.replace("model__", ""): py(v) for k, v in search.best_params_.items()},
    })
    searches[name] = search
    part = pd.DataFrame(cv)
    part.insert(0, "model", name)
    all_cv.append(part)

pd.concat(all_cv).to_csv(ROOT / "reports" / "tuning_results.csv", index=False)
res = pd.DataFrame(rows)
default_pr = cv5.groupby("model")["pr_auc"].mean()
res["default_pr_auc"] = res["model"].map(default_pr)
display(res[["model", "pr_auc", "pr_auc_std", "roc_auc", "recall_top10", "default_pr_auc"]]
        .round(3).sort_values("pr_auc", ascending=False))
for r in rows:
    print(f"{r['model']}: PR-AUC per fold {r['fold_pr_auc']}\n    {r['best_params']}")
print("\nNote: the search is scored on the same folds it tunes on, so these scores are slightly "
      "optimistic. The untouched test set in section 7 is the honest measure.")

In [ ]:
best_pr = res["pr_auc"].max()
tied = res[res["pr_auc"] >= best_pr - TIE_MARGIN].sort_values("recall_top10", ascending=False)
chosen_name = tied.iloc[0]["model"]
chosen = searches[chosen_name]
chosen_row = res[res["model"] == chosen_name].iloc[0]
print(f"Candidates within {TIE_MARGIN} PR-AUC of the best ({best_pr:.3f}): {', '.join(tied['model'])}")
print(f"Selected model: {chosen_name}")

# Out-of-fold scores of the selected configuration (each fold fitted without its own validation window)
oof = []
for i, (fit_idx, val_idx) in enumerate(folds, 1):
    pipe = clone(chosen.best_estimator_)
    pipe.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
    oof.append(pd.DataFrame({"fold": i, "y": y_train.iloc[val_idx].to_numpy(),
                             "score": pipe.predict_proba(X_train.iloc[val_idx])[:, 1]}))
oof = pd.concat(oof, ignore_index=True)

BUDGETS = [0.05, 0.10, 0.20, 0.30]
biz = []
for frac in BUDGETS:
    vals = np.array([top_k_metrics(g["y"], g["score"], frac) for _, g in oof.groupby("fold")])
    p = vals[:, 0].mean()
    biz.append({"orders_flagged_pct": int(frac * 100), "precision": p, "recall": vals[:, 1].mean(),
                "lift": vals[:, 2].mean(), "false_alarms_per_catch": (1 - p) / p if p > 0 else np.nan})
print("\nBusiness trade-off: how many orders can the marketplace act on? (3 validation folds)")
display(pd.DataFrame(biz).round(3))

**Risk tiers.** The tier cut-offs are learned from the out-of-fold validation scores: **High** = riskiest 10%, **Medium** = next 20%, **Low** = the rest. The observed late rate of each tier is reported, so the app can say "orders in this tier were late X% of the time" without overclaiming.

**Probabilities.** Class weighting inflates the raw model scores, so an isotonic calibrator is fitted on the out-of-fold scores (training data only) to convert a score into an estimated late probability. Section 7 checks how well this works on the test set.

In [ ]:
MEDIUM_Q, HIGH_Q = 0.70, 0.90
medium_cut = float(np.quantile(oof["score"], MEDIUM_Q))
high_cut = float(np.quantile(oof["score"], HIGH_Q))


def to_tier(scores):
    scores = np.asarray(scores)
    return np.where(scores >= high_cut, "High", np.where(scores >= medium_cut, "Medium", "Low"))


tiers = (oof.assign(tier=to_tier(oof["score"])).groupby("tier")["y"]
         .agg(orders="size", late_orders="sum", late_rate="mean").reindex(["High", "Medium", "Low"]))
tiers["late_rate_pct"] = (tiers["late_rate"] * 100).round(2)
print(f"Medium cut-off: score >= {medium_cut:.4f}    High cut-off: score >= {high_cut:.4f}")
display(tiers[["orders", "late_orders", "late_rate_pct"]])
print(f"Overall validation late rate: {oof['y'].mean() * 100:.2f}%")

calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
calibrator.fit(oof["score"], oof["y"])

final_pipe = chosen.best_estimator_      # already refitted on the full training set
joblib.dump(final_pipe, MODEL_PATH, compress=3)
joblib.dump(calibrator, MODELS_DIR / "calibrator.joblib")

meta = {
    "model_name": chosen_name,
    "best_params": chosen_row["best_params"],
    "selection_rule": f"PR-AUC within {TIE_MARGIN} of best, then highest recall in the riskiest 10%",
    "cv_metrics": {"pr_auc_mean": chosen_row["pr_auc"], "pr_auc_std": chosen_row["pr_auc_std"],
                   "roc_auc_mean": chosen_row["roc_auc"], "recall_top10_mean": chosen_row["recall_top10"]},
    "risk_tiers": {"medium_cutoff": medium_cut, "high_cutoff": high_cut,
                   "validation_late_rate_pct": {t: float(tiers.loc[t, "late_rate_pct"]) for t in tiers.index}},
    "train": {"orders": len(train), "late_rate_pct": round(float(y_train.mean() * 100), 2),
              "first_purchase": str(train["order_purchase_timestamp"].min().date()),
              "last_purchase": str(train["order_purchase_timestamp"].max().date())},
    "test_start": str(TEST_START.date()),
    "trained_on_test_set": False,
    "model_features": MODEL_FEATURES,
    "versions": {"python": platform.python_version(),
                 **{p: version(p) for p in ("scikit-learn", "xgboost", "pandas", "numpy")}},
}
(MODELS_DIR / "model_metadata.json").write_text(json.dumps(meta, indent=2, default=py), encoding="utf-8")

reloaded = joblib.load(MODEL_PATH)
same = np.allclose(final_pipe.predict_proba(X_train.head(500)), reloaded.predict_proba(X_train.head(500)))
print(f"\nSaved {MODEL_PATH.name}: {MODEL_PATH.stat().st_size / 1_048_576:.1f} MB   "
      f"reload check: {'OK' if same else 'FAIL'}")

## 7. Model evaluation on the untouched test set

This is the first and only time the test set (orders purchased from 2018-06-01) is used to score the model. **Nothing is tuned on it.** The two alert cut-offs come from the *validation* scores in section 6.

**Reading the results:** the test period has a lower late rate than the training period, and PR-AUC scales with the late rate. A random guesser would score a PR-AUC equal to the test late rate (3.61%), so compare every PR-AUC value with that baseline, not with the validation numbers.

In [ ]:
final = joblib.load(MODEL_PATH)                       # the saved artifact, exactly as the app will use it
dummy = build_pipeline("Dummy (prior)").fit(X_train, y_train)
proba = final.predict_proba(X_test)[:, 1]
proba_dummy = dummy.predict_proba(X_test)[:, 1]
prev = y_test.mean()
print(f"Selected model: {chosen_name}")
print(f"Test orders: {len(y_test):,}   late: {int(y_test.sum()):,}   late rate: {prev * 100:.2f}%")

metrics = {
    "model": chosen_name,
    "test_orders": int(len(y_test)), "test_late": int(y_test.sum()), "test_late_rate_pct": round(prev * 100, 2),
    "pr_auc": float(average_precision_score(y_test, proba)),
    "roc_auc": float(roc_auc_score(y_test, proba)),
    "pr_auc_dummy": float(average_precision_score(y_test, proba_dummy)),
    "roc_auc_dummy": float(roc_auc_score(y_test, proba_dummy)),
}
display(pd.DataFrame({"Selected model": [metrics["pr_auc"], metrics["roc_auc"]],
                      "Dummy baseline": [metrics["pr_auc_dummy"], metrics["roc_auc_dummy"]]},
                     index=["PR-AUC", "ROC-AUC"]).round(3))
print(f"PR-AUC is {metrics['pr_auc'] / prev:.2f}x the random-guess baseline ({prev:.3f}).")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
pr_p, pr_r, _ = precision_recall_curve(y_test, proba)
axes[0].plot(pr_r, pr_p, color=BLUE, label=f"model (PR-AUC {metrics['pr_auc']:.3f})")
axes[0].axhline(prev, color="red", ls="--", label=f"random guess ({prev:.3f})")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].set_title("Precision-recall curve (test)")
axes[0].legend()
fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, color=BLUE, label=f"model (ROC-AUC {metrics['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], color="red", ls="--", label="random guess")
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve (test)"); axes[1].legend()
finish(fig, "27_pr_roc_curves.png")

### 7.1 Confusion matrix, precision, recall, F1 and classification report

Two operating points, both fixed **before** looking at the test set: the *High* cut-off (the riskiest 10% on validation) and the *Medium* cut-off (the riskiest 30% on validation). Because the test period is calmer, the share of test orders flagged can differ from 10% and 30%.

In [ ]:
operating = {"High risk (validation top 10%)": high_cut, "Medium+ (validation top 30%)": medium_cut}
op_rows, fig_axes = [], plt.subplots(1, 2, figsize=(11, 4.5))
fig, axes = fig_axes
for ax, (label, cut) in zip(axes, operating.items()):
    pred = (proba >= cut).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["On-time", "Late"],
                                            ax=ax, colorbar=False, values_format="d", cmap="Blues")
    ax.set_title(label)
    op_rows.append({"operating point": label, "cut-off": round(cut, 4),
                    "orders flagged %": round(pred.mean() * 100, 1),
                    "precision": precision_score(y_test, pred, zero_division=0),
                    "recall": recall_score(y_test, pred, zero_division=0),
                    "F1": f1_score(y_test, pred, zero_division=0)})
finish(fig, "28_confusion_matrices.png")
op = pd.DataFrame(op_rows)
display(op.round(3))
metrics["operating_points"] = op.round(4).to_dict(orient="records")

for label, cut in operating.items():
    print(f"\nClassification report: {label}")
    print(classification_report(y_test, (proba >= cut).astype(int),
                                target_names=["On-time", "Late"], digits=3, zero_division=0))

In [ ]:
test_biz = []
for frac in BUDGETS:
    p, r, lift = top_k_metrics(y_test, proba, frac)
    test_biz.append({"orders_flagged_pct": int(frac * 100), "precision": p, "recall": r, "lift": lift,
                     "false_alarms_per_catch": (1 - p) / p if p > 0 else np.nan})
print("Test set: what acting on the riskiest X% of orders buys")
display(pd.DataFrame(test_biz).round(3))
metrics["budget_table"] = pd.DataFrame(test_biz).round(4).to_dict(orient="records")

### 7.2 Probability calibration and risk tiers on the test set

In [ ]:
cal = calibrator.predict(proba)
const = np.full(len(y_test), y_train.mean())
print(f"Test late rate:                    {prev * 100:.2f}%")
print(f"Mean calibrated probability:       {cal.mean() * 100:.2f}%")
print(f"Brier score, calibrated model:     {brier_score_loss(y_test, cal):.4f}")
print(f"Brier score, constant train rate:  {brier_score_loss(y_test, const):.4f}  (lower is better)")

tier_test = pd.DataFrame({"tier": to_tier(proba), "y": y_test.to_numpy(), "prob": cal})
tt = (tier_test.groupby("tier").agg(orders=("y", "size"), late_orders=("y", "sum"),
                                    late_rate=("y", "mean"), mean_calibrated_prob=("prob", "mean"))
      .reindex(["High", "Medium", "Low"]))
tt["late_rate_pct"] = (tt["late_rate"] * 100).round(2)
tt["mean_calibrated_prob_pct"] = (tt["mean_calibrated_prob"] * 100).round(2)
tt["share_of_late_orders_pct"] = (tt["late_orders"] / tt["late_orders"].sum() * 100).round(1)
print("\nObserved late rate in each risk tier (test) against the validation rate used in the app:")
display(tt[["orders", "late_orders", "late_rate_pct", "mean_calibrated_prob_pct", "share_of_late_orders_pct"]])
metrics["tiers_test"] = tt[["orders", "late_orders", "late_rate_pct", "mean_calibrated_prob_pct",
                            "share_of_late_orders_pct"]].reset_index().to_dict(orient="records")

frac_pos, mean_pred = calibration_curve(y_test, cal, n_bins=8, strategy="quantile")
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(mean_pred, frac_pos, "o-", color=BLUE, label="calibrated model")
lim = max(mean_pred.max(), frac_pos.max()) * 1.1
ax.plot([0, lim], [0, lim], "r--", label="perfect calibration")
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed late rate")
ax.set_title("Calibration on the test set"); ax.legend()
finish(fig, "29_calibration.png")

### 7.3 Feature importance

Permutation importance on the test set: each original feature is shuffled in turn and the drop in PR-AUC is measured (the whole pipeline is used, so one-hot columns are grouped back into their original feature). A value near zero means the model barely uses that feature.

In [ ]:
perm = permutation_importance(final, X_test, y_test, scoring="average_precision",
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
imp = (pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean,
                     "std": perm.importances_std})
       .sort_values("importance", ascending=False).reset_index(drop=True))
display(imp.round(4))
top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["feature"], top["importance"], xerr=top["std"], color=BLUE)
ax.set_xlabel("Drop in PR-AUC when the feature is shuffled")
ax.set_title("Permutation importance (test set)")
finish(fig, "30_feature_importance.png")

### 7.4 Error analysis

At the **High-risk** cut-off every test order falls into one of four outcomes: true positive (flagged and late), false positive (flagged, on time), false negative (late but missed) and true negative. We look at who is missed, who is wrongly flagged and where the model works worst.

In [ ]:
res_df = test[["order_id", "customer_state", "seller_state", "product_category", "distance_km",
               "estimated_delivery_days", "total_freight", "order_purchase_timestamp", TARGET]].copy()
res_df["score"] = proba
res_df["flag"] = (proba >= high_cut).astype(int)
res_df["tp"] = ((res_df["flag"] == 1) & (res_df[TARGET] == 1)).astype(int)
res_df["outcome"] = np.select(
    [(res_df["flag"] == 1) & (res_df[TARGET] == 1),
     (res_df["flag"] == 1) & (res_df[TARGET] == 0),
     (res_df["flag"] == 0) & (res_df[TARGET] == 1)],
    ["True positive", "False positive", "False negative"], "True negative")
display(res_df["outcome"].value_counts().to_frame("orders"))

print("Average feature values by outcome:")
display(res_df.groupby("outcome")[["distance_km", "estimated_delivery_days", "total_freight", "score"]]
        .mean().round(2))


def group_perf(col, min_late=15, top=12):
    g = res_df.groupby(col).agg(orders=(TARGET, "size"), late=(TARGET, "sum"),
                                flagged=("flag", "sum"), caught=("tp", "sum"))
    g = g[g["late"] >= min_late].copy()
    g["late_rate_pct"] = (g["late"] / g["orders"] * 100).round(2)
    g["recall"] = (g["caught"] / g["late"]).round(3)
    g["precision"] = (g["caught"] / g["flagged"].replace(0, np.nan)).round(3)
    return g.sort_values("late", ascending=False).head(top)


print("\nPerformance by customer state (states with >= 15 late test orders):")
display(group_perf("customer_state"))
print("Performance by seller state:")
display(group_perf("seller_state", min_late=10))
res_df["month"] = res_df["order_purchase_timestamp"].dt.strftime("%Y-%m")
print("Performance by purchase month (test period):")
display(group_perf("month", min_late=1))

res_df["distance_band"] = pd.cut(res_df["distance_km"], [-1, 100, 300, 600, 1000, 10000],
                                 labels=["<100 km", "100-300", "300-600", "600-1000", ">1000 km"])
print("Performance by distance band:")
display(group_perf("distance_band", min_late=1))

print("Most confident false alarms (flagged High, arrived on time):")
display(res_df[res_df["outcome"] == "False positive"].sort_values("score", ascending=False)
        [["customer_state", "seller_state", "product_category", "distance_km",
          "estimated_delivery_days", "score"]].head(8).round(3))

**What to look for in the tables above.** (a) whether missed late orders are systematically different from caught ones (distance, promised days); (b) whether recall differs strongly between states or months; (c) whether the model's behaviour in the calmer test months explains the share of false alarms. The written conclusions for section 7 are added after the results are reviewed.

## 8. Business insights

The model is used as a **ranking tool**: it sorts orders by delivery risk so a limited operations budget is spent where it matters most. The statements below about sellers, states and categories are checked on *later* data, not only on the period that produced them.

In [ ]:
# Do the sellers, states and categories that looked risky in training stay risky in the test period?
s_tr = train.groupby("seller_id")[TARGET].agg(orders_train="size", late_train="mean")
s_te = test.groupby("seller_id")[TARGET].agg(orders_test="size", late_test="mean")
joined = s_tr[s_tr["orders_train"] >= 30].join(s_te[s_te["orders_test"] >= 10], how="inner")
print(f"Sellers with >= 30 training orders and >= 10 test orders: {len(joined)}")
print(f"Spearman correlation of seller late rate, train vs test: "
      f"{joined['late_train'].corr(joined['late_test'], method='spearman'):.3f}")

cut_hi = joined["late_train"].quantile(0.90)
risky = joined[joined["late_train"] >= cut_hi]
rest = joined[joined["late_train"] < cut_hi]
pooled = lambda d: (d["late_test"] * d["orders_test"]).sum() / d["orders_test"].sum() * 100
print(f"Highest-risk 10% of sellers in training (late rate >= {cut_hi * 100:.1f}%): "
      f"test late rate {pooled(risky):.2f}%  (other sellers: {pooled(rest):.2f}%)")

worst_sellers = (s_tr[s_tr["orders_train"] >= 30].sort_values("late_train", ascending=False).head(10)
                 .join(s_te, how="left"))
worst_sellers.index = worst_sellers.index.str[:8] + "..."
worst_sellers[["late_train", "late_test"]] = (worst_sellers[["late_train", "late_test"]] * 100).round(1)
print("\nTen highest-risk sellers in training and what happened to them in the test period:")
display(worst_sellers)

for col, min_train in [("customer_state", 200), ("product_category", 300)]:
    a = train.groupby(col)[TARGET].agg(orders_train="size", late_train_pct="mean")
    b = test.groupby(col)[TARGET].agg(orders_test="size", late_test_pct="mean")
    t = a[a["orders_train"] >= min_train].join(b, how="inner")
    t[["late_train_pct", "late_test_pct"]] = (t[["late_train_pct", "late_test_pct"]] * 100).round(2)
    print(f"\n{col}: highest training late rates and their rate in the test period")
    display(t.sort_values("late_train_pct", ascending=False).head(8))

### 8.1 Recommended actions

These follow from the exploratory findings in section 3 and are tied to the model's risk tiers:

1. **Act on the High tier first.** Contact the customer proactively, reset the expectation or prioritise the shipment. The business trade-off table in sections 6 and 7 shows how many late orders are caught and how many false alarms come with each alert budget; choose the budget to match the team's capacity and the cost of a bad review.
2. **Review promised delivery windows for long-distance routes.** Late rates are highest where distance is large and the promised window is short (section 3.3). Lengthening the promise for those routes, or offering a faster option, targets the main driver.
3. **Run a seller performance programme.** Share late-rate dashboards with sellers, set a threshold for escalation and support those whose rate stays high over time. Section 8 above shows how far seller late rates persist from one period to the next, which tells how much weight to give seller history.
4. **Plan capacity around demand peaks and disruptions.** The late rate jumps in specific weeks (November 2017, February–March 2018). Early-warning monitoring of the weekly late rate lets operations add capacity or warn customers before a wave reaches its peak.
5. **Invest in the north and north-east routes** (regional carriers, regional stock), where late rates are consistently well above the national average.

### 8.2 Limitations

- **Modest predictive signal.** The model ranks risk better than chance, but precision is low, so it is a prioritisation tool, not an oracle. Many late orders cannot be predicted from purchase-time data alone.
- **Time shift.** The test period (June–August 2018) has a lower late rate than the training period, and the promised delivery windows changed. Performance and probability calibration may change again on future data, so the model should be monitored and retrained regularly.
- **Missing drivers.** The data has no weather, carrier, strike, holiday, warehouse-capacity or seller-stock information, which probably cause many delays.
- **Selection bias in the labels.** Only delivered orders with a delivery date can be labelled; canceled or lost orders are excluded.
- **Simplifications.** Multi-seller orders (1.3%) are attributed to the first seller; distances are straight-line distances between zip-code centres.
- **One marketplace, 2016–2018.** The model should not be assumed to transfer to another company or period.
- **Seller identity is not a model input.** Seller-level risk is shown for monitoring but is not used for prediction.

## 9. Files exported for the Streamlit application

The app loads the saved pipeline instead of retraining. It also needs a few **small** data files for the analytics dashboard, because the raw CSVs are far too large for GitHub and are not available on the server. They are created here.

In [ ]:
APP_DATA = ROOT / "app" / "data"
APP_DATA.mkdir(parents=True, exist_ok=True)

# 1. Test-set predictions (never seen during training) for the risk-distribution charts
dash = test[["order_purchase_timestamp", "customer_state", "seller_state", "product_category",
             "distance_km", "estimated_delivery_days", "total_price", "total_freight", TARGET]].copy()
dash["score"] = proba
dash["probability"] = cal
dash["risk_tier"] = to_tier(proba)
dash.to_csv(APP_DATA / "dashboard_test_predictions.csv", index=False)

# 2. Delivery statistics from the whole labelled table (descriptive, no model involved)
by_state = df.groupby("customer_state").agg(
    orders=(TARGET, "size"), late_rate_pct=(TARGET, "mean"),
    avg_actual_days=("actual_delivery_days", "mean"),
    avg_promised_days=("estimated_delivery_days", "mean")).reset_index()
by_state["late_rate_pct"] *= 100
by_cat = df.groupby("product_category").agg(
    orders=(TARGET, "size"), late_rate_pct=(TARGET, "mean"),
    avg_actual_days=("actual_delivery_days", "mean")).reset_index()
by_cat["late_rate_pct"] *= 100
by_month = df.groupby("purchase_year_month").agg(
    orders=(TARGET, "size"), late_rate_pct=(TARGET, "mean")).reset_index()
by_month["late_rate_pct"] *= 100
by_seller = df.groupby("seller_id").agg(
    orders=(TARGET, "size"), late_orders=(TARGET, "sum"), late_rate_pct=(TARGET, "mean"),
    seller_state=("seller_state", "first"), avg_distance_km=("distance_km", "mean")).reset_index()
by_seller["late_rate_pct"] *= 100
by_seller = by_seller[by_seller["orders"] >= 30]
for name, t in [("stats_by_customer_state", by_state), ("stats_by_category", by_cat),
                ("stats_by_month", by_month), ("stats_by_seller", by_seller)]:
    t.round(3).to_csv(APP_DATA / f"{name}.csv", index=False)

# 3. Importance and metrics for the Home / About pages
imp.round(5).to_csv(APP_DATA / "feature_importance.csv", index=False)
metrics["cv_pr_auc_mean"] = float(chosen_row["pr_auc"])
metrics["cv_pr_auc_std"] = float(chosen_row["pr_auc_std"])
metrics["model_comparison_cv"] = (cv5.groupby("model")[["pr_auc", "roc_auc", "recall_top10"]]
                                  .mean().round(4).reset_index().to_dict(orient="records"))
metrics["tier_cutoffs"] = {"medium": medium_cut, "high": high_cut}
(APP_DATA / "metrics.json").write_text(json.dumps(metrics, indent=2, default=py), encoding="utf-8")

# 4. Values for the single-prediction form (allowed choices and typical defaults from TRAINING data)
form_defaults = {
    "categorical": {c: sorted(X_train[c].dropna().unique().tolist())
                    for c in ["customer_state", "seller_state", "product_category", "payment_type"]},
    "numeric_median": {c: float(X_train[c].median()) for c in NUMERIC_MODEL},
    "numeric_p01": {c: float(X_train[c].quantile(0.01)) for c in NUMERIC_MODEL},
    "numeric_p99": {c: float(X_train[c].quantile(0.99)) for c in NUMERIC_MODEL},
}
(APP_DATA / "form_defaults.json").write_text(json.dumps(form_defaults, indent=2), encoding="utf-8")

print("Exported files:")
for p in sorted(APP_DATA.glob("*")):
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size / 1024:.0f} KB)")
for p in [MODEL_PATH, MODELS_DIR / "calibrator.joblib", MODELS_DIR / "model_metadata.json"]:
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size / 1024:.0f} KB)")

### Final checklist

- The notebook ran from top to bottom with **Restart & Run All** and no errors.
- All leakage checks in section 4 show `[OK]`.
- The test set was used only in sections 7 to 9.
- `models/late_delivery_pipeline.joblib` and the files in `app/data/` exist and are small enough for GitHub (each file far below 100 MB).

Next: build the Streamlit app, push to GitHub and deploy it on Streamlit Community Cloud.